In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pyodbc
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

print("🚀 Customer Churn Prediction - Data Extraction")
print("=============================================")


🚀 Customer Churn Prediction - Data Extraction


In [7]:
def connect_to_dwh():
    connection_strings = [
        'DRIVER={SQL Server};SERVER=(localdb)\\MSSQLLocalDB;DATABASE=CustomerChurn_DWH;Trusted_Connection=yes;',
        'DRIVER={ODBC Driver 17 for SQL Server};SERVER=(localdb)\\MSSQLLocalDB;DATABASE=CustomerChurn_DWH;Trusted_Connection=yes;',
        'DRIVER={ODBC Driver 18 for SQL Server};SERVER=(localdb)\\MSSQLLocalDB;DATABASE=CustomerChurn_DWH;Trusted_Connection=yes;TrustServerCertificate=yes;',
        'DRIVER={SQL Server};SERVER=(LocalDB)\\MSSQLLocalDB;DATABASE=CustomerChurn_DWH;Integrated Security=true;'
    ]
    
    for i, conn_str in enumerate(connection_strings, 1):
        try:
            conn = pyodbc.connect(conn_str, timeout=10)
            print(f"✅ Connected using method {i}")
            return conn
        except Exception as e:
            print(f"❌ Method {i} failed: {str(e)[:100]}")
    return None


In [8]:
connection = connect_to_dwh()
if connection:
    # Your successful query from earlier
    query = """
    SELECT 
        c.Customer_ID, c.Gender, c.Senior_Citizen, c.Partner, c.Dependents,
        c.Contract_Type, c.Churn_Flag AS Target,
        p.Internet_Service_Type, p.Phone_Service, p.Multiple_Lines,
        p.Tech_Support, p.Online_Security, p.Online_Backup, p.Device_Protection,
        p.Streaming_TV, p.Streaming_Movies, p.Plan_Price_Tier,
        pm.Payment_Method, pm.Paperless_Billing, pm.Auto_Payment,
        f.Monthly_Charges, f.Total_Charges, f.Tenure_Months,
        f.Internet_Usage_GB, f.Calls_Minutes, f.Customer_Satisfaction_Score
    FROM DWH.DimCustomer c
    JOIN DWH.FactCustomerActivity f ON c.Customer_ID = f.Customer_ID
    JOIN DWH.DimPlan p ON f.Plan_ID = p.Plan_ID
    JOIN DWH.DimPaymentMethod pm ON f.PaymentMethod_ID = pm.PaymentMethod_ID
    """
    df = pd.read_sql_query(query, connection)
    connection.close()
    print(f"✅ Data loaded from DWH: {df.shape}")
else:
    # Fallback to CSV files from Data/Raw/
    print("🔄 Using fallback CSV method...")
    try:
        customer_df = pd.read_csv('../Data/Raw/Dim_Customer.csv')
        plan_df = pd.read_csv('../Data/Raw/Dim_Plan.csv')
        payment_df = pd.read_csv('../Data/Raw/Dim_PaymentMethod.csv')
        activity_df = pd.read_csv('../Data/Raw/Fact_Customer_Activity.csv')
        
        # Join tables
        df = activity_df.merge(customer_df, on='Customer_ID')
        df = df.merge(plan_df, on='Plan_ID')
        df = df.merge(payment_df, on='PaymentMethod_ID')
        
        if 'Churn_Flag' in df.columns:
            df['Target'] = df['Churn_Flag']
            
        print("✅ Data loaded from CSV files!")
        print(f"Dataset shape: {df.shape}")
    except Exception as e:
        print(f"❌ Fallback failed: {e}")
        exit()



❌ Method 1 failed: ('08001', '[08001] [Microsoft][ODBC SQL Server Driver][DBNETLIB]SQL Server does not exist or access 
✅ Connected using method 2
✅ Data loaded from DWH: (7032, 26)


In [9]:
print("\n📊 Dataset Overview:")
print(f"Records: {len(df)}")
print(f"Features: {len(df.columns)}")
print(f"Churn Rate: {df['Target'].mean()*100:.2f}%")
print("\nMissing Values:", df.isnull().sum().sum())

print("\n📋 Data Types:")
print(df.dtypes.value_counts())

print("\n👀 First 3 rows:")
display(df.head(3))



📊 Dataset Overview:
Records: 7032
Features: 26
Churn Rate: 26.58%

Missing Values: 0

📋 Data Types:
bool       13
object      7
float64     5
int64       1
Name: count, dtype: int64

👀 First 3 rows:


,Customer_ID,Gender,Senior_Citizen,Partner,Dependents,Contract_Type,Target,Internet_Service_Type,Phone_Service,Multiple_Lines,...,Plan_Price_Tier,Payment_Method,Paperless_Billing,Auto_Payment,Monthly_Charges,Total_Charges,Tenure_Months,Internet_Usage_GB,Calls_Minutes,Customer_Satisfaction_Score
0,C000001,female,False,True,False,month-to-month,False,dsl,no,False,...,Standard,electronic check,False,False,29.85,29.85,1,11.44,14.0,7.7
1,C000002,male,False,False,False,one year,False,dsl,yes,False,...,Standard,mailed check,True,False,56.95,1889.50,34,16.39,161.0,8.5
2,C000003,male,False,False,False,month-to-month,True,dsl,yes,False,...,Standard,mailed check,True,False,53.85,108.15,2,19.39,394.0,4.8


In [10]:
df.to_csv('../Data/Processed/churn_ml_dataset.csv', index=False)
print("💾 Data saved to: ../Data/Processed/churn_ml_dataset.csv")

print("\n🎯 DATA EXTRACTION COMPLETED!")

💾 Data saved to: ../Data/Processed/churn_ml_dataset.csv

🎯 DATA EXTRACTION COMPLETED!
